In [3]:
import optuna
import pandas as pd
import matplotlib.pyplot as plt
import random
import numpy as np
from ga_core import *

In [4]:
target = load_target_image("girl_pearl_earing.png")

In [5]:
def tournament_selection(population, fitnesses, tournament_size=5):
    candidate_indices = random.sample(
        range(len(population)),
        tournament_size
    )

    best_index = min(
        candidate_indices,
        key=lambda idx: fitnesses[idx]
    )

    return population[best_index]


def roulette_selection(population, fitnesses):
    """
    Roulette selection adapted for minimization.
    Lower RMSE should have higher probability.
    """
    fitnesses = np.array(fitnesses, dtype=np.float64)

    adjusted = 1 / (fitnesses + 1e-8)
    probabilities = adjusted / adjusted.sum()

    selected_index = np.random.choice(
        len(population),
        p=probabilities
    )

    return population[selected_index]


def select_parent(
    population,
    fitnesses,
    selection_type="tournament",
    tournament_size=5
):
    if selection_type == "tournament":
        return tournament_selection(
            population,
            fitnesses,
            tournament_size
        )

    if selection_type == "roulette":
        return roulette_selection(
            population,
            fitnesses
        )

    raise ValueError(f"Unknown selection type: {selection_type}")

In [6]:
def one_point_crossover(parent_1, parent_2):
    split = random.randint(1, NUM_TRIANGLES - 1)

    child = parent_1[:split] + parent_2[split:]

    return [triangle.copy() for triangle in child]


def uniform_crossover(parent_1, parent_2):
    child = []

    for tri_1, tri_2 in zip(parent_1, parent_2):
        if random.random() < 0.5:
            child.append(tri_1.copy())
        else:
            child.append(tri_2.copy())

    return child


def apply_crossover(
    parent_1,
    parent_2,
    crossover_type="one_point"
):
    if crossover_type == "one_point":
        return one_point_crossover(parent_1, parent_2)

    if crossover_type == "uniform":
        return uniform_crossover(parent_1, parent_2)

    raise ValueError(f"Unknown crossover type: {crossover_type}")

In [7]:
def evolve_configurable(
    target_img,
    pop_size=30,
    generations=300,
    mutation_rate=0.05,
    elite_size=3,
    selection_type="tournament",
    tournament_size=5,
    crossover_type="one_point",
    crossover_rate=0.9,
    use_mutation_decay=False,
    min_mutation_rate=0.01,
    print_every=None
):
    """
    Runs a configurable Genetic Algorithm.
    This version is designed for Optuna experiments.
    """
    background_color = tuple(
        np.mean(target_img.reshape(-1, 3), axis=0).astype(int)
    )

    population = [
        create_random_individual(target_img)
        for _ in range(pop_size)
    ]

    best_fitness_history = []

    best_individual = None

    for gen in range(generations):

        rendered_images = [
            render_individual(ind, background_color)
            for ind in population
        ]

        fitnesses = [
            calculate_fitness(rendered, target_img)
            for rendered in rendered_images
        ]

        sorted_indices = np.argsort(fitnesses)

        population = [
            population[i]
            for i in sorted_indices
        ]

        fitnesses = [
            fitnesses[i]
            for i in sorted_indices
        ]

        current_best_fit = fitnesses[0]
        best_fitness_history.append(current_best_fit)
        best_individual = population[0]

        new_population = [
            individual.copy()
            for individual in population[:elite_size]
        ]

        if use_mutation_decay:
            current_mutation_rate = max(
                min_mutation_rate,
                mutation_rate * (1 - gen / generations)
            )
        else:
            current_mutation_rate = mutation_rate

        while len(new_population) < pop_size:

            parent_1 = select_parent(
                population,
                fitnesses,
                selection_type=selection_type,
                tournament_size=tournament_size
            )

            parent_2 = select_parent(
                population,
                fitnesses,
                selection_type=selection_type,
                tournament_size=tournament_size
            )

            if random.random() < crossover_rate:
                child = apply_crossover(
                    parent_1,
                    parent_2,
                    crossover_type=crossover_type
                )
            else:
                child = [triangle.copy() for triangle in parent_1]

            child = mutate(
                child,
                target_img,
                current_mutation_rate
            )

            new_population.append(child)

        population = new_population

        if print_every is not None:
            if gen % print_every == 0 or gen == generations - 1:
                print(
                    f"Generation {gen:04d} | "
                    f"Best RMSE: {current_best_fit:.4f}"
                )

    return best_individual, best_fitness_history, background_color

In [8]:
OPTUNA_GENERATIONS = 1000
N_TRIALS = 30

In [9]:
def objective(trial):
    """
    Searches for the best combination of GA components.
    """

    pop_size = trial.suggest_categorical(
        "pop_size",
        [20, 30, 40]
    )

    mutation_rate = trial.suggest_float(
        "mutation_rate",
        0.005,
        0.12
    )

    use_mutation_decay = trial.suggest_categorical(
        "use_mutation_decay",
        [False, True]
    )

    elite_size = trial.suggest_int(
        "elite_size",
        1,
        max(2, int(pop_size * 0.2))
    )

    selection_type = trial.suggest_categorical(
        "selection_type",
        ["tournament", "roulette"]
    )

    if selection_type == "tournament":
        tournament_size = trial.suggest_int(
            "tournament_size",
            2,
            min(8, pop_size)
        )
    else:
        tournament_size = 2

    crossover_type = trial.suggest_categorical(
        "crossover_type",
        ["one_point", "uniform"]
    )

    crossover_rate = trial.suggest_float(
        "crossover_rate",
        0.60,
        1.00
    )

    best_ind, history, background_color = evolve_configurable(
        target_img=target,
        pop_size=pop_size,
        generations=OPTUNA_GENERATIONS,
        mutation_rate=mutation_rate,
        elite_size=elite_size,
        selection_type=selection_type,
        tournament_size=tournament_size,
        crossover_type=crossover_type,
        crossover_rate=crossover_rate,
        use_mutation_decay=use_mutation_decay,
        min_mutation_rate=0.01,
        print_every=None
    )

    best_rend = render_individual(
        best_ind,
        background_color
    )

    final_rmse = calculate_fitness(
        best_rend,
        target
    )

    return final_rmse

In [ ]:
study = optuna.create_study(
    direction="minimize",
    study_name="full_ga_component_optimization"
)

study.optimize(
    objective,
    n_trials=N_TRIALS
)

[I 2026-05-19 12:38:11,955] A new study created in memory with name: full_ga_component_optimization


In [ ]:
optuna_results_df = study.trials_dataframe()

display_columns = [
    "number",
    "value",
    "params_pop_size",
    "params_mutation_rate",
    "params_use_mutation_decay",
    "params_elite_size",
    "params_selection_type",
    "params_tournament_size",
    "params_crossover_type",
    "params_crossover_rate",
    "state"
]

available_columns = [
    col for col in display_columns
    if col in optuna_results_df.columns
]

optuna_results_df[available_columns]

In [ ]:
plt.figure(figsize=(8, 4))

plt.plot(
    optuna_results_df["number"],
    optuna_results_df["value"],
    marker="o"
)

plt.title("Optuna Search: GA Component Optimization")
plt.xlabel("Trial")
plt.ylabel("Final RMSE")
plt.grid(True)

plt.show()

In [ ]:
best_params = study.best_params

best_pop_size = best_params["pop_size"]
best_mutation_rate = best_params["mutation_rate"]
best_use_mutation_decay = best_params["use_mutation_decay"]
best_elite_size = best_params["elite_size"]
best_selection_type = best_params["selection_type"]
best_crossover_type = best_params["crossover_type"]
best_crossover_rate = best_params["crossover_rate"]

if best_selection_type == "tournament":
    best_tournament_size = best_params["tournament_size"]
else:
    best_tournament_size = 2

best_ind_optuna, history_optuna, background_color_optuna = evolve_configurable(
    target_img=target,
    pop_size=best_pop_size,
    generations=1000,
    mutation_rate=best_mutation_rate,
    elite_size=best_elite_size,
    selection_type=best_selection_type,
    tournament_size=best_tournament_size,
    crossover_type=best_crossover_type,
    crossover_rate=best_crossover_rate,
    use_mutation_decay=best_use_mutation_decay,
    min_mutation_rate=0.01,
    print_every=100
)

best_rend_optuna = render_individual(
    best_ind_optuna,
    background_color_optuna
)

final_rmse_optuna = calculate_fitness(
    best_rend_optuna,
    target
)

print(f"Final RMSE with Optuna configuration: {final_rmse_optuna:.4f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    1,
    2,
    figsize=(10, 5)
)

ax1.imshow(target)
ax1.set_title("Original")
ax1.axis("off")

ax2.imshow(best_rend_optuna)
ax2.set_title(
    f"Optuna Best Configuration\nRMSE = {final_rmse_optuna:.2f}"
)
ax2.axis("off")

plt.tight_layout()
plt.show()